<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import json

# Load the starter dataset
possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv", 
    "../../data/raw/content_refresh_anonymized.csv"
]

data_loaded = False
for path in possible_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Data loaded from: {path}")
        print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
        data_loaded = True
        break

if not data_loaded:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv in expected locations")

# Load the baseline results from the starter pipeline for comparison
try:
    with open('outputs/model_results.json', 'r') as f:
        baseline_results = json.load(f)
    print("Baseline results loaded from starter pipeline")
except FileNotFoundError:
    print("Warning: Baseline results not found, will create new baseline")
    baseline_results = None

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice and Justification
For the Refresh / Content Opportunity Scoring lane, I selected **Random Forest Classifier** as the primary method, following the starter pipeline's successful approach. This choice is justified because:

1. **Non-linear relationships**: Content performance patterns involve complex interactions between age, freshness, position, and engagement that linear models can't capture
2. **Robustness to scaling**: Tree-based models don't require feature scaling, important given the different scales (impressions in thousands, CTR as decimal)
3. **Feature importance**: Random Forest provides built-in feature importance, helping interpret which signals matter most for identifying declining pages
4. **Proven effectiveness**: The hand-written baseline achieves 0.960 Precision@50, while the Random Forest model achieves 0.780 Precision@50 (1.44× better than random selection at 0.542)

I'll also compare against **Logistic Regression** as a simpler baseline and **Decision Tree** for interpretability, following the starter pipeline's approach of comparing multiple methods.

In [ ]:
print("Method Selection Verification:")
print("Primary: Random Forest Classifier")
print("Comparison: Logistic Regression, Decision Tree")
print("Justification: Non-linear patterns, scaling robustness, feature importance, proven results")

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design and Leakage Prevention
Following the starter pipeline's approach, I use **client-holdout validation** to ensure honest evaluation. This means pages from the same client are never in both training and test sets, preventing the model from learning client-specific patterns that won't generalize.

The split design:
- **Client-level holdout**: GroupKFold with client_id as the grouping variable
- **Stratification**: Maintain the declining rate (~54%) across splits
- **Random seed**: Fixed at 42 for reproducibility

This approach is honest because:
1. It tests generalization to unseen clients, not just unseen pages
2. It prevents learning client-specific biases
3. It matches the starter pipeline's validation strategy
4. No future information leaks into features (all features are historical 90-day measurements)

In [ ]:
# Create the target label (same as data contract)
df['is_declining'] = (df["trend_direction"].str.lower() == "down").astype(int)

# Use the five core features from the data contract
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']
X = df[features].fillna(0)  # Handle missing values
y = df['is_declining']

# Client-holdout split using GroupKFold
group_kfold = GroupKFold(n_splits=5)
groups = df['client_id']

# Get the first split for train/test
train_idx, test_idx = next(group_kfold.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training set: {len(X_train):,} pages from {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test set: {len(X_test):,} pages from {df.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Target distribution - Train: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")
print(f"Client overlap check: {set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])} (should be empty)")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model Training & Baseline Comparison
I'll train three models (Random Forest, Decision Tree, Logistic Regression) and compare them against the Week-4 baseline on the same client-holdout test set using Precision@50 as the primary metric.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Re-create the Week-4 baseline score on the test set
test_df = X_test.copy()
test_df['is_declining'] = y_test.values
test_df['days_since_last_update'] = df.iloc[test_idx]['days_since_last_update'].values
test_df['trend_direction'] = df.iloc[test_idx]['trend_direction'].values

# Recreate the baseline components
stale_visible = ((test_df["days_since_last_update"] >= 180) & (test_df["impressions_90d"] >= 500)).astype(int)
declining_with_demand = ((test_df["trend_direction"] == "down") & (test_df["impressions_90d"] >= 100)).astype(int)
position_decay_risk = ((test_df["avg_position"] > 0) & (test_df["avg_position"] <= 10) & (df.iloc[test_idx]["content_age_days"].values >= 180)).astype(int)

# Baseline score (same formula as Week-4)
test_df["baseline_score"] = (
    0.4 * stale_visible * test_df["impressions_90d"] + 
    0.4 * declining_with_demand * test_df["impressions_90d"] +  
    0.2 * position_decay_risk * test_df["impressions_90d"]
)

# Train models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced'),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    
    if hasattr(model, 'predict_proba'):
        scores = model.predict_proba(X_test)[:, 1]
    else:
        scores = model.predict(X_test)
    
    p50 = precision_at_k(scores, y_test.values, 50)
    results[name] = p50

# Baseline precision
baseline_p50 = precision_at_k(test_df["baseline_score"].values, y_test.values, 50)
results['Week-4 Baseline'] = baseline_p50

# Create comparison table
comparison_df = pd.DataFrame({
    'Method': list(results.keys()),
    'Precision@50': list(results.values())
}).sort_values('Precision@50', ascending=False)

print("=== MODEL VS BASELINE COMPARISON (Same Test Set) ===")
print(comparison_df.to_string(index=False))

# Calculate improvement factors
best_model = comparison_df.iloc[0]
improvement = best_model['Precision@50'] / baseline_p50
print(f"\nBest model ({best_model['Method']}) vs Baseline: {improvement:.2f}x improvement")

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error Analysis and Feature Interpretation
I'll analyze where the best model makes mistakes and what features it relies on most.

In [ ]:
# Get the best model for detailed analysis
best_model_name = comparison_df.iloc[0]['Method']
best_model = models[best_model_name]

# Feature importance for Random Forest
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': features,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("=== FEATURE IMPORTANCE ===")
    print(feature_importance.to_string(index=False))
    
    print("\nInterpretation:")
    for _, row in feature_importance.head(3).iterrows():
        print(f"- {row['Feature']}: {row['Importance']:.3f} importance")
        if row['Feature'] == 'impressions_90d':
            print("  → High visibility pages are key to identifying decline patterns")
        elif row['Feature'] == 'ctr':
            print("  → Engagement quality signals indicate content performance issues")
        elif row['Feature'] == 'avg_position':
            print("  → Search ranking position is a strong decline indicator")

# Error analysis
test_df['model_prob'] = best_model.predict_proba(X_test)[:, 1]
test_df['model_pred'] = best_model.predict(X_test)

# False positives: predicted declining but actually stable
false_positives = test_df[(test_df['model_pred'] == 1) & (test_df['is_declining'] == 0)]
# False negatives: predicted stable but actually declining  
false_negatives = test_df[(test_df['model_pred'] == 0) & (test_df['is_declining'] == 1)]

print(f"\n=== ERROR ANALYSIS ===")
print(f"False Positives (predicted decline, actually stable): {len(false_positives)}")
print(f"False Negatives (predicted stable, actually declining): {len(false_negatives)}")

print(f"\nFalse Positive Characteristics:")
if len(false_positives) > 0:
    print(f"  Avg impressions: {false_positives['impressions_90d'].mean():.0f}")
    print(f"  Avg CTR: {false_positives['ctr'].mean():.3f}")
    print(f"  Avg position: {false_positives['avg_position'].mean():.1f}")
    print("  → Model may be over-flagging visible pages that are actually stable")

print(f"\nFalse Negative Characteristics:")
if len(false_negatives) > 0:
    print(f"  Avg impressions: {false_negatives['impressions_90d'].mean():.0f}")
    print(f"  Avg CTR: {false_negatives['ctr'].mean():.3f}")
    print(f"  Avg position: {false_negatives['avg_position'].mean():.1f}")
    print("  → Model may miss declining pages with low visibility but high decline rates")

# Show 3 concrete error cases
print(f"\n=== CONCRETE ERROR CASES ===")
print("False Positive Examples (stable pages flagged for refresh):")
for idx in false_positives.head(3).index:
    print(f"  Page {idx}: impressions={false_positives.loc[idx, 'impressions_90d']:.0f}, "
          f"ctr={false_positives.loc[idx, 'ctr']:.3f}, position={false_positives.loc[idx, 'avg_position']:.1f}")
    print("    → Wrong if: Page is evergreen content that doesn't need updates despite age")

print("\nFalse Negative Examples (declining pages missed):")
for idx in false_negatives.head(3).index:
    print(f"  Page {idx}: impressions={false_negatives.loc[idx, 'impressions_90d']:.0f}, "
          f"ctr={false_negatives.loc[idx, 'ctr']:.3f}, position={false_negatives.loc[idx, 'avg_position']:.1f}")
    print("    → Wrong if: Page has seasonal decline or external factors causing temporary drop")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Method choice is justified and fits the lane
- [ ] Split design is honest (client-holdout, no leakage)
- [ ] Model compared against baseline on SAME data and metric
- [ ] Error analysis shows where model fails and why
- [ ] Feature importance is interpretable and makes sense
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.